# Phase 6 SP1 Validation

This notebook runs Python-side relation checks and optional Rust/SP1 checks.


In [ ]:
script = 'from __future__ import annotations\n\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\n\nSCRIPT_PATH = Path(__file__).resolve()\nDEFAULT_REPO_NAME = "zk_offline_dqn"\nDEFAULT_REPO_URL = "https://github.com/patee1811/zk_offline_dqn.git"\nDEFAULT_REPO_BRANCH = "cleanup-project-structure"\nSUMMARY_PATH = Path("artifacts/reports/kaggle_sp1_validation_summary.json")\n\n\ndef tail(text: str, lines: int = 40) -> str:\n    return "\\n".join(text.splitlines()[-lines:])\n\n\ndef run_command(command: List[str], cwd: Path, timeout: Optional[int] = None) -> Dict[str, Any]:\n    started = time.perf_counter()\n    env = os.environ.copy()\n    env["PYTHONPATH"] = str(cwd)\n    env.setdefault("MPLBACKEND", "Agg")\n    try:\n        result = subprocess.run(\n            command,\n            cwd=cwd,\n            env=env,\n            text=True,\n            capture_output=True,\n            timeout=timeout,\n        )\n        returncode = result.returncode\n        stdout = result.stdout\n        stderr = result.stderr\n        status = "passed" if returncode == 0 else "failed"\n    except subprocess.TimeoutExpired as exc:\n        returncode = None\n        stdout = exc.stdout or ""\n        stderr = exc.stderr or ""\n        status = "timeout"\n    except Exception as exc:  # pragma: no cover - environment dependent\n        returncode = None\n        stdout = ""\n        stderr = repr(exc)\n        status = "error"\n    elapsed = time.perf_counter() - started\n    record = {\n        "command": command,\n        "cwd": cwd.as_posix(),\n        "returncode": returncode,\n        "duration_sec": round(elapsed, 4),\n        "stdout_tail": tail(stdout),\n        "stderr_tail": tail(stderr),\n        "status": status,\n    }\n    print("command =", " ".join(command))\n    print("status =", status)\n    print("returncode =", returncode)\n    print("duration_sec =", record["duration_sec"])\n    if record["stdout_tail"]:\n        print("stdout_tail =")\n        print(record["stdout_tail"])\n    if record["stderr_tail"]:\n        print("stderr_tail =")\n        print(record["stderr_tail"])\n    print()\n    return record\n\n\ndef looks_like_repo(path: Path) -> bool:\n    return (\n        (path / "zk_offline_dqn").is_dir()\n        and (path / "scripts/experiments").is_dir()\n        and (path / "zk_backend/test_vectors/td_mvp_case_0.json").exists()\n    )\n\n\ndef clone_repo_if_requested(target: Path) -> Optional[Dict[str, Any]]:\n    repo_url = (\n        os.environ.get("ZK_OFFLINE_DQN_REPO_URL")\n        or os.environ.get("REPO_URL")\n        or DEFAULT_REPO_URL\n    )\n    repo_branch = os.environ.get("ZK_OFFLINE_DQN_REPO_BRANCH") or DEFAULT_REPO_BRANCH\n    if not repo_url:\n        return None\n    if target.exists():\n        return {\n            "command": ["git", "clone", repo_url, target.as_posix()],\n            "cwd": target.parent.as_posix(),\n            "returncode": 0,\n            "duration_sec": 0.0,\n            "stdout_tail": f"target already exists: {target}",\n            "stderr_tail": "",\n            "status": "skipped_existing_target",\n        }\n    command = ["git", "clone"]\n    if repo_branch:\n        command.extend(["-b", repo_branch])\n    command.extend([repo_url, target.as_posix()])\n    return run_command(command, cwd=target.parent)\n\n\ndef ensure_phase6_support_files(repo_root: Path) -> None:\n    diagnostic = repo_root / "scripts/experiments/check_sp1_environment.py"\n    if diagnostic.exists():\n        return\n    diagnostic.parent.mkdir(parents=True, exist_ok=True)\n    diagnostic.write_text(\n        "\\n".join(\n            [\n                "from __future__ import annotations",\n                "import platform, shutil, subprocess, sys",\n                "from pathlib import Path",\n                "def check(cmd):",\n                "    exe = shutil.which(cmd[0])",\n                "    if exe is None:",\n                "        print(cmd[0] + \'_available = False\')",\n                "        return",\n                "    result = subprocess.run(cmd, text=True, capture_output=True)",\n                "    print(cmd[0].replace(\' \', \'_\') + \'_available = \' + str(result.returncode == 0))",\n                "    if result.stdout.strip():",\n                "        print(cmd[0] + \'_stdout = \' + result.stdout.strip().splitlines()[0])",\n                "    if result.stderr.strip():",\n                "        print(cmd[0] + \'_stderr = \' + result.stderr.strip().splitlines()[0])",\n                "print(\'=== SP1 ENVIRONMENT DIAGNOSTIC ===\')",\n                "print(\'python = \' + sys.version)",\n                "print(\'platform = \' + platform.platform())",\n                "print(\'cwd = \' + Path.cwd().as_posix())",\n                "print(\'is_kaggle = \' + str(Path(\'/kaggle\').exists()))",\n                "print(\'sp1_dir_exists = \' + str(Path(\'zk_backend/td_mvp/sp1\').exists()))",\n                "check([\'rustc\', \'--version\'])",\n                "check([\'cargo\', \'--version\'])",\n                "check([\'cargo\', \'prove\', \'--version\'])",\n                "check([\'sp1up\', \'--version\'])",\n            ]\n        )\n        + "\\n",\n        encoding="utf-8",\n    )\n\n\ndef discover_repo_root() -> tuple[Optional[Path], List[Dict[str, Any]]]:\n    events: List[Dict[str, Any]] = []\n    candidates = [\n        Path.cwd(),\n        SCRIPT_PATH.parents[2] if len(SCRIPT_PATH.parents) > 2 else SCRIPT_PATH.parent,\n        Path("/kaggle/working/zk_offline_dqn"),\n        Path("/kaggle/working") / DEFAULT_REPO_NAME,\n    ]\n    for candidate in candidates:\n        if looks_like_repo(candidate):\n            return candidate.resolve(), events\n\n    clone_target = Path("/kaggle/working") / DEFAULT_REPO_NAME\n    if Path("/kaggle/working").exists():\n        event = clone_repo_if_requested(clone_target)\n        if event is not None:\n            events.append(event)\n        if looks_like_repo(clone_target):\n            return clone_target.resolve(), events\n\n    return None, events\n\n\ndef inspect_package_name(path: Path) -> Optional[str]:\n    if not path.exists():\n        return None\n    for line in path.read_text(encoding="utf-8").splitlines():\n        stripped = line.strip()\n        if stripped.startswith("name = "):\n            return stripped.split("=", 1)[1].strip().strip(\'"\')\n    return None\n\n\ndef cargo_available() -> bool:\n    return shutil.which("cargo") is not None\n\n\ndef main() -> int:\n    print("=== KAGGLE SP1 VALIDATION ===")\n    print("python =", sys.version)\n    print("platform =", platform.platform())\n    print("cwd =", Path.cwd().as_posix())\n    print("is_kaggle_path =", Path("/kaggle").exists())\n    print()\n\n    repo_root, discovery_events = discover_repo_root()\n    commands: List[Dict[str, Any]] = list(discovery_events)\n    if repo_root is None:\n        summary = {\n            "status": "repo_not_found",\n            "python": sys.version,\n            "platform": platform.platform(),\n            "cwd": Path.cwd().as_posix(),\n            "commands": commands,\n            "message": (\n                "Could not locate repository files. Upload this repository to the "\n                "Kaggle kernel working directory or set ZK_OFFLINE_DQN_REPO_URL."\n            ),\n        }\n        SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)\n        SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")\n        print(json.dumps(summary, indent=2))\n        return 0\n\n    print("repo_root =", repo_root.as_posix())\n    ensure_phase6_support_files(repo_root)\n    py = sys.executable\n    python_commands = [\n        [py, "-m", "compileall", "zk_offline_dqn", "scripts", "src", "tests"],\n        [py, "-m", "unittest", "discover", "tests/regression"],\n        [\n            py,\n            "-m",\n            "zk_offline_dqn.cli.main",\n            "verify",\n            "td-mvp",\n            "--input",\n            "zk_backend/test_vectors/td_mvp_case_0.json",\n        ],\n        [\n            py,\n            "scripts/experiments/benchmark_distinct_td_sp1.py",\n            "--skip-sp1",\n            "--out-dir",\n            "artifacts/benchmarks/distinct_td_sp1_python_smoke",\n        ],\n        [\n            py,\n            "scripts/experiments/benchmark_forward_td_mlp_sp1.py",\n            "--skip-sp1",\n            "--out-dir",\n            "artifacts/benchmarks/forward_td_mlp_sp1_python_smoke",\n        ],\n        [\n            py,\n            "scripts/experiments/benchmark_one_step_sgd_tiny_sp1.py",\n            "--skip-sp1",\n            "--out-dir",\n            "artifacts/benchmarks/one_step_sgd_tiny_sp1_python_smoke",\n        ],\n        [py, "scripts/experiments/check_sp1_environment.py"],\n    ]\n    python_results: List[Dict[str, Any]] = []\n    for command in python_commands:\n        result = run_command(command, cwd=repo_root)\n        python_results.append(result)\n        commands.append(result)\n\n    sp1_dir = repo_root / "zk_backend/td_mvp/sp1"\n    host_package = inspect_package_name(sp1_dir / "host/Cargo.toml")\n    guest_package = inspect_package_name(sp1_dir / "guest/Cargo.toml")\n    shared_package = inspect_package_name(sp1_dir / "shared/Cargo.toml")\n    cargo_status = {\n        "cargo_available": cargo_available(),\n        "sp1_dir_exists": sp1_dir.exists(),\n        "host_package": host_package,\n        "guest_package": guest_package,\n        "shared_package": shared_package,\n    }\n\n    if cargo_status["cargo_available"] and sp1_dir.exists():\n        commands.append(run_command(["cargo", "test"], cwd=sp1_dir, timeout=1800))\n        if host_package:\n            execute_command = [\n                "cargo",\n                "run",\n                "--release",\n                "-p",\n                host_package,\n                "--",\n                "--execute",\n            ]\n            commands.append(run_command(execute_command, cwd=sp1_dir, timeout=1800))\n            if os.environ.get("RUN_SP1_PROVE") == "1":\n                prove_command = [\n                    "cargo",\n                    "run",\n                    "--release",\n                    "-p",\n                    host_package,\n                    "--",\n                    "--prove",\n                ]\n                commands.append(run_command(prove_command, cwd=sp1_dir, timeout=7200))\n            else:\n                commands.append(\n                    {\n                        "command": ["RUN_SP1_PROVE=1", "cargo", "run", "--release", "-p", host_package, "--", "--prove"],\n                        "cwd": sp1_dir.as_posix(),\n                        "returncode": None,\n                        "duration_sec": 0.0,\n                        "stdout_tail": "Proof not run because RUN_SP1_PROVE is not set to 1.",\n                        "stderr_tail": "",\n                        "status": "skipped",\n                    }\n                )\n    else:\n        commands.append(\n            {\n                "command": ["cargo", "test"],\n                "cwd": sp1_dir.as_posix(),\n                "returncode": None,\n                "duration_sec": 0.0,\n                "stdout_tail": "Cargo/SP1 workspace unavailable; Rust/SP1 commands skipped.",\n                "stderr_tail": "",\n                "status": "skipped",\n            }\n        )\n\n    summary = {\n        "status": "completed",\n        "python": sys.version,\n        "platform": platform.platform(),\n        "cwd": Path.cwd().as_posix(),\n        "repo_root": repo_root.as_posix(),\n        "cargo_status": cargo_status,\n        "commands": commands,\n        "all_required_python_passed": all(item["status"] == "passed" for item in python_results),\n        "sp1_prove_requested": os.environ.get("RUN_SP1_PROVE") == "1",\n    }\n    output_path = repo_root / SUMMARY_PATH\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    output_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")\n    print("summary_path =", output_path.as_posix())\n    print(json.dumps(summary, indent=2, ensure_ascii=False))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
namespace = {'__name__': 'phase6_embedded', '__file__': 'kaggle_sp1_validation_embedded.py'}
exec(compile(script, 'kaggle_sp1_validation_embedded.py', 'exec'), namespace)
namespace['main']()
